# Auto Label
This notebook is used to:
- Automatically generate a caption and a label for each image using a free model
- I'm too broke for any openai API calls :(( so i'll just use Qwen-2.5-VL-3B model

- Then, we can use the generated labels to **train a smaller model or a BERT for better accuracy**
- Manually correct mislabled image afterwards, check **data_cleaning.ipynb** for more info

In [6]:
!source ../miniconda3/bin/activate
!conda --version

conda 24.11.3


In [18]:
import os
import json
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

# --------------------------------------------------
# Configuration
# --------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_DIR = "fine_tune_dataset/val/images"
OUTPUT_CSV = "val_labels.csv"

In [19]:
# --------------------------------------------------
# Load processor and model
# --------------------------------------------------
# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print("Warning: Running on CPU may be slow. GPU is recommended.")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="cuda",                 # 🧠 This spreads across GPUs
    torch_dtype=torch.bfloat16,         # Use bfloat16 for lower memory
)
print("Successfully loaded the model")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Successfully loaded the model


In [20]:
import re, json
# Directory of images
records = []

# Fire analysis prompt
system_prompt = (
    "You are a visual analyst evaluating an image for signs of fire and the surrounding context. "
    "Perform the following tasks:\n"
    "1: Summarize what you see in the image. Describe the environment, key objects, people, and any signs of fire or smoke.\n"
    "2: Based on your summary, classify the fire situation into one of the following three categories:\n"
    "- **No fire**: The image may contain symbols, representations, or objects related to fire, but no actual fire or smoke is present. This includes things like warning signs, fire safety equipment (extinguishers, alarms), digital or printed representations of fire (e.g., a laptop screen showing fire, a drawing or painting), or thematic decorations.\n"
    "- **Controlled fire**: There is a visible flame or heat source, but it is clearly contained, expected, and managed by people or objects in its environment. This includes fireplaces, campfires, cooking appliances, candles, lighters, or matchsticks. There should be no signs of danger, spread, panic, or damage in the surroundings.\n"
    "- **Dangerous (uncontrolled) fire**: The fire appears out of control or harmful to the environment or people. Signs include flames spreading to flammable materials (e.g., curtains, furniture, bedding), thick smoke near the ceiling, charring, visible structural damage, or people reacting with fear or urgency. The context suggests potential or ongoing property damage or bodily harm.\n"
    "Return only this JSON format:\n"
    "{ \"caption\": \"...\", \"label\": \"no fire\"|\"controlled fire\"|\"dangerous fire\" }"
)

# Loop through image files
for fname in os.listdir(IMAGE_DIR):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    image_path = os.path.join(IMAGE_DIR, fname)
    try:
        image = Image.open(image_path).convert("RGB")

        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": system_prompt}
            ]
        }]

        # Generate prompt and inputs
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt").to(device)

        # Run model
        generate_ids = model.generate(**inputs, max_new_tokens=1024)
        output = processor.tokenizer.batch_decode(generate_ids, skip_special_tokens=True)[0]
        if output.startswith("system"):
            output = output.split("assistant", 1)[-1].strip()
        # Clean up any system/user/assistant roles
        clean_output = re.sub(r'^(system|user|assistant)\n', '', output, flags=re.MULTILINE).strip()

        # Match the last complete JSON block
        matches = re.findall(r'\{[^{}]+\}', clean_output, re.DOTALL)
        if matches:
            try:
                result = json.loads(matches[-1])  # take the last match (most likely correct)
                caption = result.get("caption", "").strip()
                label = result.get("label", "").strip()
            except Exception as e:
                print(f"❌ Failed to parse JSON for {fname}: {e}")
                caption = clean_output
                label = "unknown"
        else:
            print(f"❌ No JSON block found in output for {fname}")
            caption = clean_output
            label = "unknown"

        print(f"✅ {fname} - {label} - {caption} ")
        records.append({
            "image_path": image_path,
            "caption": caption,
            "label": label
        })

    except Exception as e:
        print(f"❌ Error processing {fname}: {e}")

✅ val_313.jpg - no fire - The image shows an aerial view of a large industrial facility surrounded by greenery and water bodies. There are two tall chimneys emitting smoke, indicating the presence of a power plant or similar facility. The area is bordered by trees and fields, with a river running through it. 
✅ val_1091.jpg - controlled fire - Two individuals are working in a kitchen with a pizza oven that has a visible flame inside. There are stacks of pizza boxes on the counter. 
✅ val_257.jpg - no fire - The image shows an aerial view of an industrial area with multiple buildings. The central building has a flat roof with several cylindrical structures, possibly chimneys or vents. The sky is partly cloudy, and there are trees and greenery in the background. 
✅ val_292.jpg - controlled fire - The image shows a controlled fire burning on the ground, with flames and smoke rising from a piece of paper or fabric that is being burned. The fire is contained within a designated area, and th

In [21]:
df = pd.DataFrame(records)[['image_path', 'label', 'caption']]  # Specify column order
df.to_csv(OUTPUT_CSV, index=False)
print(f"📁 Saved results to {OUTPUT_CSV}")

📁 Saved results to val_labels.csv
